# SPiCe Tutorial: Modelling Epithelial-Mesenchymal Plasticity

This notebook demonstrates the full SPiCe workflow for breast cancer Xenium
spatial transcriptomics data using the `spice.tl` / `spice.pl` API.

All tool functions modify `adata` in place — results are stored in
`adata.obs`, `adata.obsm` and `adata.uns['spice']`.

In [1]:
import os
os.environ["PYTHONWARNINGS"] = "ignore"

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import numpy as np
import pandas as pd
import scanpy as sc
import enrichmap as em
import spice

In [4]:
sc.set_figure_params(frameon=False)

In [5]:
project_dir = "/home/cenk/Withnell_spice/"
os.chdir(project_dir)

figures_dir = project_dir +"figures/"

## 1. Load data into AnnData

In [6]:
adata = sc.read_h5ad("processed_data/xenium_breast_annotated.h5ad")

In [ ]:
import decoupler as dc

hallmark = dc.op.hallmark()
emt_signature = hallmark[hallmark["source"] == "EPITHELIAL_MESENCHYMAL_TRANSITION"]["target"].values
emt_dict = {"EMT": emt_signature}

In [ ]:
em.tl.score(adata, emt_dict)

In [ ]:
adata.obs["tumour_cells"] = np.where(adata.obs["cell_type"] == "Tumour", "tumour", "non-tumour")

## 2. Assign discrete EMT state labels

In [ ]:
spice.tl.assign_state_labels(
    adata,
    score_key="EMT_score",
    label_key="state_label",
    n_states=4,
    tumour_mask_key="tumour_cells",
)
print(adata.obs["state_label"].value_counts())

## 3. Build the spatial graph

In [ ]:
spice.tl.build_graph(
    adata,
    spatial_key="spatial",
    n_neighbors=12,
    celltype_key="cell_type",
    label_key="state_label",
    score_key="EMT_score",
)

G = adata.uns["spice"]["graph"]
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## 4. Cross-validate

In [ ]:
spice.tl.cross_validate(
    adata,
    feature_mode="celltype",
    num_folds=5,
    num_epochs=500,
    hidden_dim1=16,
    hidden_dim2=16,
    dropout=0.58,
    learning_rate=0.009,
)

## 5. Evaluate

In [ ]:
spice.tl.evaluate(adata)
adata.uns["spice"]["auc"]

## 6. Plot AUC per class

In [ ]:
EMT_MAP = {0: "EPI", 1: "H-EPI", 2: "H-MES", 3: "MES"}
spice.pl.auc_per_class(adata, state_map=EMT_MAP, save=figures_dir + "breast_auc_per_class_hallmark.pdf")

In [ ]:
spice.tl.explain_nodes(adata, n_explanations=300)  # increase to 300 for publication quality
spice.pl.node_importance(adata, state_map=EMT_MAP, save=figures_dir + "breast_node_importance.pdf")

In [ ]:
spice.tl.sanitize(adata)
adata.write_h5ad("processed_data/xenium_breast_hallmark_spice_results.h5ad")